In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kgan31/doha-data/kavitakosh_dohas_split.csv


In [5]:
CSV_PATH = "/kaggle/input/datasets/kgan31/doha-data/kavitakosh_dohas_split.csv"

In [6]:
"""
N-Gram Language Model for Hindi Doha Generation
================================================
Builds character-level and word-level N-gram models on your doha dataset.
Includes:
  - Unigram, Bigram, Trigram, and higher-order N-gram models
  - Laplace (add-k) smoothing
  - Kneser-Ney smoothing
  - Perplexity evaluation
  - BLEU score evaluation
  - Text generation via N-gram sampling

Kaggle setup:
    CSV_PATH = "/kaggle/input/doha-dataset/doha_dataset.csv"

Local setup:
    pip install pandas numpy nltk
"""

import re
import math
import random
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction
import nltk
nltk.download("punkt", quiet=True)

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────

# CSV_PATH    = "doha_dataset.csv"   # ← change to your path
N           = 3                    # Default N-gram order (try 2, 3, 4)
SMOOTHING_K = 1.0                  # Laplace smoothing constant (add-k)
VAL_SPLIT   = 0.1                  # Fraction held out for evaluation
GEN_LENGTH  = 120                  # Max characters/words to generate
TEMPERATURE = 0.8                  # Sampling temperature
SEED        = 42

random.seed(SEED)
np.random.seed(SEED)


# ─────────────────────────────────────────────────────────────────────────────
# 1. DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────

def clean_doha(text: str) -> str:
    """Remove verse markers and normalize whitespace."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"।।\s*\d+\s*।।", "।।", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def load_dataset(csv_path: str):
    """Load CSV and return cleaned doha list and full corpus string."""
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    df = df.dropna(subset=["Doha"]).reset_index(drop=True)
    df["Doha_clean"] = df["Doha"].apply(clean_doha)

    doha_list = df["Doha_clean"].tolist()
    corpus    = "\n".join(doha_list)

    print(f"✅ Loaded {len(doha_list)} dohas")
    print(f"   Total characters : {len(corpus):,}")
    return doha_list, corpus, df


def split_data(doha_list: list, val_split: float = VAL_SPLIT):
    """Split doha list into train and validation sets."""
    random.shuffle(doha_list)
    split      = int(len(doha_list) * (1 - val_split))
    train_list = doha_list[:split]
    val_list   = doha_list[split:]
    print(f"   Train dohas : {len(train_list)}")
    print(f"   Val dohas   : {len(val_list)}\n")
    return train_list, val_list


# ─────────────────────────────────────────────────────────────────────────────
# 2. TOKENIZATION
# ─────────────────────────────────────────────────────────────────────────────

# Special tokens
BOS = "<BOS>"    # Beginning of sequence
EOS = "<EOS>"    # End of sequence
UNK = "<UNK>"    # Unknown token


def char_tokenize(text: str) -> list:
    """
    Character-level tokenization.
    Each Devanagari character (including matras) is one token.
    Preserves spaces as tokens for readability.
    """
    return list(text)


def word_tokenize_hindi(text: str) -> list:
    """
    Word-level tokenization for Hindi/Braj text.
    Splits on whitespace and Devanagari punctuation.
    """
    text   = re.sub(r"[।॥]", " । ", text)
    tokens = text.split()
    tokens = [t for t in tokens if t.strip()]
    return tokens


def tokenize_corpus(doha_list: list, mode: str = "char") -> list:
    """
    Tokenize all dohas and wrap each with BOS/EOS markers.

    Args:
        doha_list : list of doha strings
        mode      : 'char' for character-level, 'word' for word-level

    Returns:
        List of token lists, one per doha
    """
    tokenizer = char_tokenize if mode == "char" else word_tokenize_hindi
    tokenized = []
    for doha in doha_list:
        tokens = [BOS] + tokenizer(doha) + [EOS]
        tokenized.append(tokens)
    return tokenized


# ─────────────────────────────────────────────────────────────────────────────
# 3. N-GRAM MODEL
# ─────────────────────────────────────────────────────────────────────────────

class NgramModel:
    """
    N-gram language model with two smoothing options:
      1. Laplace (add-k) smoothing  — simple, works for small N
      2. Kneser-Ney smoothing       — state-of-the-art for N-gram models

    Supports both character-level and word-level tokenization.

    Usage:
        model = NgramModel(n=3, smoothing='laplace', mode='char')
        model.fit(train_dohas)
        ppl = model.perplexity(val_dohas)
        text = model.generate(seed="मोर")
    """

    def __init__(self, n: int = 3, smoothing: str = "laplace",
                 k: float = 1.0, mode: str = "char"):
        """
        Args:
            n         : N-gram order (1=unigram, 2=bigram, 3=trigram, ...)
            smoothing : 'laplace' or 'kneser_ney'
            k         : Laplace smoothing constant (ignored for kneser_ney)
            mode      : 'char' or 'word'
        """
        assert n >= 1, "N must be >= 1"
        assert smoothing in ("laplace", "kneser_ney"), \
            "smoothing must be 'laplace' or 'kneser_ney'"

        self.n         = n
        self.smoothing = smoothing
        self.k         = k
        self.mode      = mode

        # Core count tables
        self.ngram_counts   = defaultdict(Counter)  # context → {token: count}
        self.unigram_counts = Counter()             # token → count
        self.vocab          = set()
        self.total_tokens   = 0

        # Kneser-Ney specific
        self.kn_continuation = Counter()    # token → #unique contexts it follows
        self.kn_total_bigrams = 0
        self.D = 0.75                       # KN discount constant

    # ── Fitting ──────────────────────────────────────────────────────────────

    def fit(self, doha_list: list):
        """
        Train the N-gram model on a list of doha strings.
        Counts all N-grams up to order N.
        """
        tokenized = tokenize_corpus(doha_list, mode=self.mode)

        for tokens in tokenized:
            for token in tokens:
                self.unigram_counts[token] += 1
                self.vocab.add(token)
                self.total_tokens += 1

            # Count N-grams of all orders up to N
            for i in range(len(tokens)):
                for order in range(2, self.n + 1):
                    if i + order <= len(tokens):
                        ngram   = tuple(tokens[i : i + order])
                        context = ngram[:-1]
                        word    = ngram[-1]
                        self.ngram_counts[context][word] += 1

        # Kneser-Ney: count continuation unigrams (bigram diversity)
        if self.smoothing == "kneser_ney":
            seen_bigrams = set()
            for tokens in tokenized:
                for i in range(len(tokens) - 1):
                    bigram = (tokens[i], tokens[i + 1])
                    if bigram not in seen_bigrams:
                        self.kn_continuation[tokens[i + 1]] += 1
                        seen_bigrams.add(bigram)
            self.kn_total_bigrams = sum(self.kn_continuation.values())

        vocab_size = len(self.vocab)
        print(f"✅ {self.n}-gram model ({self.smoothing}) fitted")
        print(f"   Vocabulary     : {vocab_size:,} tokens")
        print(f"   Total tokens   : {self.total_tokens:,}")
        print(f"   Unique contexts: {len(self.ngram_counts):,}")

    # ── Probability ──────────────────────────────────────────────────────────

    def prob_laplace(self, token: str, context: tuple) -> float:
        """
        P(token | context) with Laplace (add-k) smoothing.

        P = (count(context, token) + k) / (count(context) + k * |V|)

        Guarantees non-zero probability for all tokens.
        Falls back to lower-order N-gram if context unseen.
        """
        V = len(self.vocab)

        if self.n == 1 or len(context) == 0:
            # Unigram
            return (self.unigram_counts.get(token, 0) + self.k) / \
                   (self.total_tokens + self.k * V)

        context_counts = self.ngram_counts.get(context, {})
        context_total  = sum(context_counts.values())

        if context_total == 0:
            # Back off to (N-1)-gram
            return self.prob_laplace(token, context[1:])

        return (context_counts.get(token, 0) + self.k) / \
               (context_total + self.k * V)

    def prob_kneser_ney(self, token: str, context: tuple) -> float:
        """
        P_KN(token | context) — interpolated Kneser-Ney smoothing.

        Uses continuation counts for lower-order distributions,
        giving better estimates for rare contexts.

        KN formula (bigram example):
          P_KN(w|h) = max(c(h,w) - D, 0) / c(h)
                    + λ(h) * P_continuation(w)

        where P_continuation(w) = |{h: c(h,w) > 0}| / total_bigrams
        """
        if self.n == 1 or len(context) == 0:
            # Base: continuation probability
            if self.kn_total_bigrams == 0:
                return 1.0 / max(len(self.vocab), 1)
            return self.kn_continuation.get(token, 0) / self.kn_total_bigrams

        context_counts = self.ngram_counts.get(context, {})
        context_total  = sum(context_counts.values())

        if context_total == 0:
            return self.prob_kneser_ney(token, context[1:])

        # Discounted count
        discounted = max(context_counts.get(token, 0) - self.D, 0) / context_total

        # Number of unique words that follow this context (for λ)
        n_unique   = len(context_counts)
        lambda_val = (self.D * n_unique) / context_total

        # Lower-order interpolation
        lower = self.prob_kneser_ney(token, context[1:])

        return discounted + lambda_val * lower

    def log_prob(self, token: str, context: tuple) -> float:
        """Return log2 probability of token given context."""
        if self.smoothing == "laplace":
            p = self.prob_laplace(token, context)
        else:
            p = self.prob_kneser_ney(token, context)
        return math.log2(max(p, 1e-10))

    # ── Perplexity ────────────────────────────────────────────────────────────

    def perplexity(self, doha_list: list) -> float:
        """
        Compute perplexity on a list of dohas.

        PP = 2^(-1/N * sum(log2 P(w_i | context_i)))

        Lower perplexity = better model.
        Interpretation:
          - Perplexity ≈ vocab_size  → model is no better than random
          - Perplexity → 1           → model perfectly predicts next token
          - Character-level Hindi: reasonable range 3–15 for trained models
        """
        tokenized  = tokenize_corpus(doha_list, mode=self.mode)
        total_log  = 0.0
        total_toks = 0

        for tokens in tokenized:
            for i in range(self.n - 1, len(tokens)):
                token   = tokens[i]
                context = tuple(tokens[max(0, i - self.n + 1) : i])
                total_log  += self.log_prob(token, context)
                total_toks += 1

        if total_toks == 0:
            return float("inf")

        avg_log = total_log / total_toks
        return 2 ** (-avg_log)

    # ── Generation ────────────────────────────────────────────────────────────

    def generate(self, seed: str = None, max_len: int = GEN_LENGTH,
                 temperature: float = TEMPERATURE) -> str:
        """
        Generate text autoregressively using the N-gram model.

        At each step:
          1. Extract the (N-1)-gram context from recent tokens
          2. Score all vocab tokens given context
          3. Apply temperature scaling
          4. Sample from the resulting distribution

        Args:
            seed        : Seed string (tokenized as start context).
                          If None, starts from BOS token.
            max_len     : Max tokens to generate
            temperature : Controls randomness (lower = more deterministic)

        Returns:
            Generated string (detokenized)
        """
        tokenizer = char_tokenize if self.mode == "char" else word_tokenize_hindi

        if seed:
            tokens = [BOS] + tokenizer(seed)
        else:
            tokens = [BOS]

        for _ in range(max_len):
            context = tuple(tokens[-(self.n - 1):]) if self.n > 1 else ()

            # Score all vocab tokens
            candidates = list(self.vocab - {BOS, UNK})
            if not candidates:
                break

            if self.smoothing == "laplace":
                log_probs = np.array([
                    self.log_prob(t, context) for t in candidates
                ])
            else:
                log_probs = np.array([
                    self.log_prob(t, context) for t in candidates
                ])

            # Temperature scaling in log space
            log_probs = log_probs / temperature
            log_probs -= log_probs.max()          # numerical stability
            probs      = np.exp(log_probs)
            probs      = probs / probs.sum()

            next_token = np.random.choice(candidates, p=probs)

            if next_token == EOS:
                break

            tokens.append(next_token)

        # Detokenize
        output_tokens = [t for t in tokens if t not in (BOS, EOS, UNK)]
        if self.mode == "char":
            return "".join(output_tokens)
        else:
            return " ".join(output_tokens)

    # ── BLEU ─────────────────────────────────────────────────────────────────

    def bleu_evaluate(self, doha_list: list, n_samples: int = 20) -> dict:
        """
        Generate n_samples dohas and compute character-level BLEU
        against the full reference corpus.

        Uses character-level BLEU because:
          - Braj/Hindi vocab overlap between generated and reference is low
          - Character n-grams capture script similarity better than words
        """
        print(f"\n📊 BLEU evaluation on {n_samples} samples...")
        smoother  = SmoothingFunction().method1
        all_scores = {f"bleu{n}": [] for n in range(1, 5)}

        for _ in range(n_samples):
            ref_doha  = random.choice(doha_list)
            seed_len  = min(6, len(ref_doha) // 3)
            seed      = ref_doha[:seed_len]

            generated = self.generate(seed=seed, max_len=GEN_LENGTH)

            hyp_chars = list(generated)
            ref_chars = [list(r) for r in doha_list]

            for n in range(1, 5):
                weights = tuple([1.0 / n] * n + [0.0] * (4 - n))
                score   = sentence_bleu(
                    ref_chars, hyp_chars,
                    weights=weights,
                    smoothing_function=smoother
                )
                all_scores[f"bleu{n}"].append(score)

        results = {k: np.mean(v) for k, v in all_scores.items()}
        for k, v in results.items():
            print(f"   {k.upper()} : {v:.4f}")
        return results


# ─────────────────────────────────────────────────────────────────────────────
# 4. COMPARISON: MULTIPLE N-GRAM ORDERS
# ─────────────────────────────────────────────────────────────────────────────

def compare_ngram_orders(train_dohas: list, val_dohas: list,
                         orders: list = [1, 2, 3, 4],
                         smoothings: list = ["laplace", "kneser_ney"],
                         mode: str = "char"):
    """
    Train and evaluate models across multiple N values and smoothing methods.
    Prints a comparison table — useful for picking the best configuration.
    """
    print(f"\n{'='*65}")
    print(f"N-GRAM COMPARISON  (mode={mode})")
    print(f"{'='*65}")
    print(f"{'N':>4} {'Smoothing':>12} {'Val PPL':>10} {'Train PPL':>11}")
    print("-" * 45)

    results = []
    for n in orders:
        for smoothing in smoothings:
            model = NgramModel(n=n, smoothing=smoothing, mode=mode)
            model.fit(train_dohas)

            train_ppl = model.perplexity(train_dohas)
            val_ppl   = model.perplexity(val_dohas)

            print(f"{n:>4} {smoothing:>12} {val_ppl:>10.2f} {train_ppl:>11.2f}")
            results.append({
                "n": n, "smoothing": smoothing,
                "train_ppl": train_ppl, "val_ppl": val_ppl
            })

    print("-" * 45)
    best = min(results, key=lambda x: x["val_ppl"])
    print(f"✅ Best config: N={best['n']}, smoothing={best['smoothing']}, "
          f"val_ppl={best['val_ppl']:.2f}")
    return results


# ─────────────────────────────────────────────────────────────────────────────
# 5. MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print(f"\n{'='*60}")
    print("N-Gram Language Model — Hindi Doha")
    print(f"{'='*60}\n")

    # ── Load data ──────────────────────────────────────────
    doha_list, corpus, df = load_dataset(CSV_PATH)
    train_dohas, val_dohas = split_data(doha_list, VAL_SPLIT)

    # ── Compare N-gram orders (char-level) ─────────────────
    compare_ngram_orders(
        train_dohas, val_dohas,
        orders=[1, 2, 3, 4],
        smoothings=["laplace", "kneser_ney"],
        mode="char"
    )

    # ── Train best model (trigram + KN, char-level) ────────
    print(f"\n{'='*60}")
    print("TRAINING BEST MODEL: Trigram + Kneser-Ney (char)")
    print(f"{'='*60}")

    best_model = NgramModel(n=N, smoothing="kneser_ney", mode="char")
    best_model.fit(train_dohas)

    train_ppl = best_model.perplexity(train_dohas)
    val_ppl   = best_model.perplexity(val_dohas)
    print(f"\n  Train Perplexity : {train_ppl:.2f}")
    print(f"  Val   Perplexity : {val_ppl:.2f}")

    # ── BLEU evaluation ────────────────────────────────────
    bleu_scores = best_model.bleu_evaluate(val_dohas, n_samples=20)

    # ── Generate sample dohas ──────────────────────────────
    print(f"\n{'='*60}")
    print("GENERATED DOHAS")
    print(f"{'='*60}")

    seeds = [d[:6] for d in val_dohas[:5]] if len(val_dohas) >= 5 else ["मोर"]
    for seed in seeds:
        out = best_model.generate(seed=seed, max_len=GEN_LENGTH, temperature=TEMPERATURE)
        print(f"\n  Seed : {seed!r}")
        print(f"  → {out}")

    # ── Word-level trigram for comparison ──────────────────
    print(f"\n{'='*60}")
    print("WORD-LEVEL TRIGRAM (for comparison)")
    print(f"{'='*60}")

    word_model = NgramModel(n=3, smoothing="kneser_ney", mode="word")
    word_model.fit(train_dohas)
    word_train_ppl = word_model.perplexity(train_dohas)
    word_val_ppl   = word_model.perplexity(val_dohas)
    print(f"\n  Train Perplexity : {word_train_ppl:.2f}")
    print(f"  Val   Perplexity : {word_val_ppl:.2f}")

    out = word_model.generate(seed=seeds[0], max_len=40)
    print(f"  Sample → {out}")

    # ── Summary table ──────────────────────────────────────
    print(f"\n{'='*60}")
    print("SUMMARY")
    print(f"{'='*60}")
    print(f"  Model           : {N}-gram Kneser-Ney (char)")
    print(f"  Train PPL       : {train_ppl:.2f}")
    print(f"  Val PPL         : {val_ppl:.2f}")
    print(f"  BLEU-1          : {bleu_scores['bleu1']:.4f}")
    print(f"  BLEU-2          : {bleu_scores['bleu2']:.4f}")
    print(f"  BLEU-4          : {bleu_scores['bleu4']:.4f}")
    print(f"{'='*60}\n")


if __name__ == "__main__":
    main()


N-Gram Language Model — Hindi Doha

✅ Loaded 8190 dohas
   Total characters : 644,844
   Train dohas : 7371
   Val dohas   : 819


N-GRAM COMPARISON  (mode=char)
   N    Smoothing    Val PPL   Train PPL
---------------------------------------------
✅ 1-gram model (laplace) fitted
   Vocabulary     : 134 tokens
   Total tokens   : 588,129
   Unique contexts: 0
   1      laplace      33.45       33.59
✅ 1-gram model (kneser_ney) fitted
   Vocabulary     : 134 tokens
   Total tokens   : 588,129
   Unique contexts: 0
   1   kneser_ney      72.16       72.09
✅ 2-gram model (laplace) fitted
   Vocabulary     : 134 tokens
   Total tokens   : 588,129
   Unique contexts: 133
   2      laplace      14.53       14.53
✅ 2-gram model (kneser_ney) fitted
   Vocabulary     : 134 tokens
   Total tokens   : 588,129
   Unique contexts: 133
   2   kneser_ney      14.37       14.35
✅ 3-gram model (laplace) fitted
   Vocabulary     : 134 tokens
   Total tokens   : 588,129
   Unique contexts: 3,043
   3   